<a href="https://colab.research.google.com/github/shenoyrg-hub/iitr_capstone/blob/main/Capstone_Project_Part_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Part B

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,mean_squared_error,mean_absolute_error,r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from google.colab import files
import os

# Create the data and outputs directories if they don't exist
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Upload your file
uploaded = files.upload()

# Move the uploaded file into the 'data' directory
for filename in uploaded.keys():
    os.rename(filename, os.path.join('data', filename))

PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no"]

# Load the modelling dataframe from the saved CSV file on disk to prevent NameErrors
modelling_df = pd.read_csv('data/modelling_table.csv')

# Set X and y
x = modelling_df[FEATURES]
y = modelling_df["delivery_hours"]

# Split into training, validation and test
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
train_rows = len(X_train)
val_rows = len(X_val)
test_rows = len(X_test)
train_scaled_max_abs_mean = abs(X_train_scaled).max(axis=0).mean()
test_scaled_max_abs_mean = abs(X_test_scaled).max(axis=0).mean()

# Baseline metrics (Modified to avoid the deprecated 'squared' argument)
val_mean = y_train.mean()
val_median = y_train.median()
val_mae = mean_absolute_error(y_val, [val_mean] * len(y_val))
val_mse = mean_squared_error(y_val, [val_mean] * len(y_val))
val_rmse = np.sqrt(val_mse)
val_r2 = r2_score(y_val, [val_mean] * len(y_val))

# Clear tree depth file if it exists to avoid appending endlessly
tree_depth_path = Path("outputs/part_b_tree_depth.csv")
if tree_depth_path.exists():
    tree_depth_path.unlink()

# Tree-depth diagnostic (Modified to avoid the deprecated 'squared' argument)
MAX_DEPTHS = [1, 2, 3, 4, 6, 8, 12, None]
for max_depth in MAX_DEPTHS:
  model = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
  model.fit(X_train, y_train)
  train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
  val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
  output_df = pd.DataFrame(
      {
          "max_depth": ["unlimited" if max_depth is None else str(max_depth)],
          "train_rmse": [train_rmse],
          "val_rmse": [val_rmse],
      }
  )

  output_df.to_csv("outputs/part_b_tree_depth.csv", mode="a", index=False, header=not tree_depth_path.exists())

# Model pipeline evaluation
model_names = [
    "mean_baseline",
    "linear_regression",
    "decision_tree",
    "random_forest",
    "gradient_boosting",
]

from sklearn.linear_model import LinearRegression
models = [
    None,
    Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    DecisionTreeRegressor(max_depth=4, random_state=42),
    RandomForestRegressor(n_estimators=100, random_state=42),
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
]

metrics_rows = []
y_pred_baseline = [val_mean] * len(y_val)
metrics_rows.append({
    "model": "mean_baseline",
    "split": "val",
    "MAE": mean_absolute_error(y_val, y_pred_baseline),
    "MSE": val_mse,
    "RMSE": val_rmse,
    "R2": val_r2
})

for name, model in zip(model_names[1:], models[1:]):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    metrics_rows.append({
        "model": name,
        "split": "val",
        "MAE": mean_absolute_error(y_val, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_val, y_pred)
    })

# Learning-Rate Experiment
lr_values = [0.01, 0.1, 1.0]
gb_lr_rows = []
for lr in lr_values:
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=42)
    gb.fit(X_train, y_train)
    y_val_pred = gb.predict(X_val)
    mse = mean_squared_error(y_val, y_val_pred)
    gb_lr_rows.append({
        "learning_rate": lr,
        "val_rmse": np.sqrt(mse),
        "val_r2": r2_score(y_val, y_val_pred)
    })

gb_lr_df = pd.DataFrame(gb_lr_rows)
gb_lr_df.to_csv("outputs/part_b_gb_learning_rate.csv", index=False)

# Cross-Validated comparison on Train + Val
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

top_candidates = {
    "gradient_boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

cv_rows = []
for model_name, model in top_candidates.items():
    scores = cross_val_score(model, X_train_val, y_train_val, cv=kf, scoring="r2")
    cv_rows.append({
        "model": model_name,
        "mean_r2": scores.mean(),
        "std_r2": scores.std()
    })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv("outputs/part_b_cv.csv", index=False)

# Final Evaluation of best model on Test split
best_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
best_model.fit(X_train, y_train)
y_test_pred = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)

metrics_rows.append({
    "model": "gradient_boosting",
    "split": "final-test",
    "MAE": mean_absolute_error(y_test, y_test_pred),
    "MSE": test_mse,
    "RMSE": np.sqrt(test_mse),
    "R2": r2_score(y_test, y_test_pred)
})

metrics_df = pd.DataFrame(metrics_rows)[["model", "split", "MAE", "MSE", "RMSE", "R2"]]
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)
print("Successfully generated all part B files & metrics.")
# Display the full comparison table in the Colab output cell
print("\n--- Model Performance Summary ---")
print(metrics_df.to_string(index=False))

Saving modelling_table.csv to modelling_table.csv
Successfully generated all part B files & metrics.

--- Model Performance Summary ---
            model      split       MAE         MSE      RMSE        R2
    mean_baseline        val 26.186675 1009.798659 36.572154 -0.188014
linear_regression        val 33.179192 1749.053822 41.821691 -1.057737
    decision_tree        val 24.159221 1005.366523 31.707515 -0.182800
    random_forest        val 26.870271 1141.273890 33.782745 -0.342693
gradient_boosting        val 30.025419 1369.295322 37.003991 -0.610957
gradient_boosting final-test 26.657504 1019.035442 31.922335  0.048514
